## Seperate converter for a new CSV for select dates

## Combines and drops duplicates in CSVs

In [ ]:

old_df = pd.read_csv("/Users/sreej/Desktop/Gateway_lawrence/gatewayinitiative-lawrencepd/scripts/data/pdfs/v2_lawrence_2018_2024.csv")
new_df = pd.read_csv("/Users/sreej/Desktop/Gateway_lawrence/gatewayinitiative-lawrencepd/scripts/data/pdfs/march_june_backfill.csv")

combined = pd.concat([old_df, new_df], ignore_index=True)

combined = combined.drop_duplicates(
    subset=["Incident #"],
    keep="first"
)

combined.to_csv(
    "combined_lawrence_2018_2024.cs",
    index=False
)

print(f"Added {len(new_df)} rows")
print(f"Final rows: {len(combined)}")

Added 6208 rows
Final rows: 437893


In [ ]:
#RUN, change to dir parameters, haven't used though
import os

notebook_dir = os.getcwd()
PDF_DATA_DIR = os.path.join(notebook_dir, 'data', 'pdfs')
parsed_csv_path = os.path.join(PDF_DATA_DIR, 'lawrence_2018_to_2022.csv')

# run for all PDFs, add optional `max_pdfs` argument
parse_all_pdfs_to_csv(PDF_DATA_DIR, parsed_csv_path)



✅ CSV created at: /Users/sreej/Desktop/Gateway_lawrence/gatewayinitiative-lawrencepd/scripts/data/pdfs/lawrence_2018_to_2022.csv


### Converting PDFs to Text from missing_pdfs (Dates that were missing)

In [ ]:
import os
import pdfplumber
import pytesseract
from pdf2image import convert_from_path
import csv
import re
from html import unescape

In [ ]:
# Patterns.  #make a missing pdfs notebook for the missing dates, access the missing_dates.txt file 

incident_pattern = r"Incident #:\s*(\d+)"
date_pattern = r"Date:\s*(\d{4}-\d{2}-\d{2}\s*\d{2}:\d{2}:\d{2})"
type_pattern = r"Type:\s*([\w\s/]+)"
location_pattern = r"Location:\s*(.+?)(?=\n|$)"
arrest_pattern = r"Arrested:"
name_pattern = r"Name:\s*([^:\n]+?)(?=\s*Date of Birth:)"
dob_pattern = r"Date of Birth:\s*(\d{2}/\d{2}/\d{4})"
charges_pattern = r"Charges:\s*((?:.+?(\n|$))*?)(?=\n(?:\w+:|$))"

def is_meaningful_police_log(text):
    incident_count = len(re.findall(r"Incident\s+#", text))
    has_keywords = "Location:" in text or "Type:" in text or "NOISE ORD" in text
    is_not_all_symbols = bool(re.search(r"[A-Za-z]{3,}", text))
    return incident_count > 0 and has_keywords and is_not_all_symbols

def extract_entry(entry):
    return {
        "Incident #": re.search(incident_pattern, entry).group(1) if re.search(incident_pattern, entry) else "",
        "Date": re.search(date_pattern, entry).group(1) if re.search(date_pattern, entry) else "",
        "Type": re.search(type_pattern, entry).group(1) if re.search(type_pattern, entry) else "",
        "Location": re.search(location_pattern, entry).group(1) if re.search(location_pattern, entry) else "",
        "Arrested": "Yes" if re.search(arrest_pattern, entry) else "No",
        "Name": re.search(name_pattern, entry).group(1).strip() if re.search(name_pattern, entry) else "",
        "DOB": re.search(dob_pattern, entry).group(1) if re.search(dob_pattern, entry) else "",
        "Charges": "; ".join(
            line.strip() for line in re.search(charges_pattern, entry, re.DOTALL).group(1).splitlines()
            if line.strip()
        ) if re.search(charges_pattern, entry, re.DOTALL) else ""
    }

def extract_data_from_text_pdfplumber(text):
    rows = []
    if not re.search(r"[=-]{10,}", text):
        return rows
    incidents = re.split(r"[=-]{10,}", text)
    for entry in incidents:
        rows.append(extract_entry(entry))
    return rows

def extract_data_from_text_ocr(text):
    rows = []
    incidents = re.split(r"(?=Incident\s+#?:\s*\d+)", text)
    for entry in incidents:
        rows.append(extract_entry(entry))
    return rows

def extract_data_from_pdf_auto(pdf_path):
    try:
        with pdfplumber.open(pdf_path) as pdf:
            text = "\n".join(p.extract_text() or "" for p in pdf.pages)
            text = unescape(text)
            if is_meaningful_police_log(text):
                return extract_data_from_text_pdfplumber(text)
            else:
                print(f"⚠️ Unreadable by pdfplumber, using OCR: {os.path.basename(pdf_path)}")
    except Exception as e:
        print(f"❌ pdfplumber failed on {pdf_path}: {e}")

    try:
        images = convert_from_path(pdf_path, dpi=300)
        ocr_text = ""
        for img in images:
            ocr_text += pytesseract.image_to_string(img)
        if is_meaningful_police_log(ocr_text):
            return extract_data_from_text_ocr(ocr_text)
    except Exception as e:
        print(f"❌ OCR failed for {pdf_path}: {e}")
    return []

def parse_all_pdfs_to_csv(input_dir, output_csv, max_pdfs=None):
    if not os.path.isdir(input_dir):
        raise FileNotFoundError(f"Input PDF directory not found: {input_dir}")

    all_rows = []
    pdf_count = 0
    for root, _, files in os.walk(input_dir):
        for file in sorted(files):
            if file.lower().endswith(".pdf"):
                pdf_path = os.path.join(root, file)
                print(f"📄 Parsing: {file}")
                rows = extract_data_from_pdf_auto(pdf_path)
                if not rows:
                    print(f"⚠️ No extractable data in: {file}")
                all_rows.extend(rows)
                pdf_count += 1
                if max_pdfs and pdf_count >= max_pdfs:
                    print(f"⏸️ Stopping after {pdf_count} PDFs")
                    break
        if max_pdfs and pdf_count >= max_pdfs:
            break

    with open(output_csv, "w", newline="", encoding="utf-8") as f:
        fieldnames = ["Incident #", "Date", "Type", "Location", "Arrested", "Name", "DOB", "Charges"]
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(all_rows)
    print(f"\n✅ CSV created at: {output_csv}")

In [ ]:
input_dir = os.path.join(notebook_dir, "data", "pdfs")  # or "data/2023_law_pd_data", etc.
output_csv = os.path.join(notebook_dir, "data", "missing_dates_csv", "unclean_missing_dates.csv")
parse_all_pdfs_to_csv(input_dir, output_csv)


In [ ]:
import os
import csv
import re
from html import unescape
from datetime import datetime, timedelta

import pdfplumber
import pytesseract
from pdf2image import convert_from_path


# ============================================================
# Paths
# ============================================================

notebook_dir = os.getcwd()

PDF_DATA_DIR = os.path.join(
    notebook_dir,
    "data",
    "pdfs"
)

EXISTING_CSV = os.path.join(
    PDF_DATA_DIR,
    "v2_lawrence_2018_2024.csv"
)

OUTPUT_CSV = os.path.join(
    PDF_DATA_DIR,
    "march_june_backfill.csv"
)

# This searches anywhere inside data/pdfs
PDF_SEARCH_ROOT = PDF_DATA_DIR


# ============================================================
# Target dates only
# ============================================================

def date_range(start, end):
    start_dt = datetime.strptime(start, "%Y-%m-%d").date()
    end_dt = datetime.strptime(end, "%Y-%m-%d").date()

    dates = []
    current = start_dt

    while current <= end_dt:
        dates.append(current)
        current += timedelta(days=1)

    return dates


TARGET_DATES = (  ### TARGET DATES INPUT FOR BACKFILL  ## fix and make seperate cell
    date_range("2019-03-01", "2019-03-13")
    + date_range("2019-06-01", "2019-06-19")
)

TARGET_FILENAMES = [
    d.strftime("%m-%d-%Y") + ".pdf"
    for d in TARGET_DATES
]

print("Target PDFs:")
for f in TARGET_FILENAMES:
    print(f)


# ============================================================
# Regex patterns
# ============================================================

incident_pattern = r"Incident #:\s*(\d+)"
date_pattern = r"Date:\s*(\d{4}-\d{2}-\d{2}\s*\d{2}:\d{2}:\d{2})"
type_pattern = r"Type:\s*([\w\s/]+)"
location_pattern = r"Location:\s*(.+?)(?=\n|$)"
arrest_pattern = r"Arrested:"
name_pattern = r"Name:\s*([^:\n]+?)(?=\s*Date of Birth:)"
dob_pattern = r"Date of Birth:\s*(\d{2}/\d{2}/\d{4})"
charges_pattern = r"Charges:\s*((?:.+?(\n|$))*?)(?=\n(?:\w+:|$))"


# ============================================================
# Extraction helpers
# ============================================================

def is_meaningful_police_log(text):
    incident_count = len(re.findall(r"Incident\s+#", text))
    has_keywords = "Location:" in text or "Type:" in text or "NOISE ORD" in text
    is_not_all_symbols = bool(re.search(r"[A-Za-z]{3,}", text))
    return incident_count > 0 and has_keywords and is_not_all_symbols


def extract_entry(entry, source_pdf):
    return {
        "Source PDF": source_pdf,
        "Incident #": re.search(incident_pattern, entry).group(1)
            if re.search(incident_pattern, entry) else "",
        "Date": re.search(date_pattern, entry).group(1)
            if re.search(date_pattern, entry) else "",
        "Type": re.search(type_pattern, entry).group(1)
            if re.search(type_pattern, entry) else "",
        "Location": re.search(location_pattern, entry).group(1)
            if re.search(location_pattern, entry) else "",
        "Arrested": "Yes" if re.search(arrest_pattern, entry) else "No",
        "Name": re.search(name_pattern, entry).group(1).strip()
            if re.search(name_pattern, entry) else "",
        "DOB": re.search(dob_pattern, entry).group(1)
            if re.search(dob_pattern, entry) else "",
        "Charges": "; ".join(
            line.strip()
            for line in re.search(charges_pattern, entry, re.DOTALL).group(1).splitlines()
            if line.strip()
        ) if re.search(charges_pattern, entry, re.DOTALL) else ""
    }


def extract_data_from_text_pdfplumber(text, source_pdf):
    rows = []

    if not re.search(r"[=-]{10,}", text):
        return rows

    incidents = re.split(r"[=-]{10,}", text)

    for entry in incidents:
        row = extract_entry(entry, source_pdf)

        if row["Incident #"] or row["Date"]:
            rows.append(row)

    return rows


def extract_data_from_text_ocr(text, source_pdf):
    rows = []

    incidents = re.split(r"(?=Incident\s+#?:\s*\d+)", text)

    for entry in incidents:
        row = extract_entry(entry, source_pdf)

        if row["Incident #"] or row["Date"]:
            rows.append(row)

    return rows


def extract_data_from_pdf_auto(pdf_path):
    source_pdf = os.path.basename(pdf_path)

    try:
        with pdfplumber.open(pdf_path) as pdf:
            text = "\n".join(page.extract_text() or "" for page in pdf.pages)
            text = unescape(text)

            if is_meaningful_police_log(text):
                print(f"   ✅ Used pdfplumber")
                return extract_data_from_text_pdfplumber(text, source_pdf)
            else:
                print(f"   ⚠️ pdfplumber unreadable, trying OCR")

    except Exception as e:
        print(f"   ❌ pdfplumber failed: {e}")

    try:
        images = convert_from_path(pdf_path, dpi=300)

        ocr_text = ""

        for img in images:
            ocr_text += pytesseract.image_to_string(img)

        ocr_text = unescape(ocr_text)

        if is_meaningful_police_log(ocr_text):
            print(f"   ✅ Used OCR")
            return extract_data_from_text_ocr(ocr_text, source_pdf)
        else:
            print(f"   ⚠️ OCR ran but did not produce meaningful police-log text")

    except Exception as e:
        print(f"   ❌ OCR failed: {e}")

    return []


# ============================================================
# Find target PDFs
# ============================================================

def find_target_pdfs(search_root, target_filenames):
    found = {}

    for root, _, files in os.walk(search_root):
        for file in files:
            if file in target_filenames:
                found[file] = os.path.join(root, file)

    return found


found_pdfs = find_target_pdfs(PDF_SEARCH_ROOT, TARGET_FILENAMES)

print()
print("====================================")
print("PDF SEARCH SUMMARY")
print("====================================")
print(f"Target PDFs expected: {len(TARGET_FILENAMES)}")
print(f"Target PDFs found:    {len(found_pdfs)}")
print(f"Target PDFs missing:  {len(TARGET_FILENAMES) - len(found_pdfs)}")

missing_files = [f for f in TARGET_FILENAMES if f not in found_pdfs]

if missing_files:
    print()
    print("Missing target PDFs:")
    for f in missing_files:
        print(f" - {f}")


# ============================================================
# Parse only target PDFs
# ============================================================

all_rows = []

print()
print("====================================")
print("PARSING TARGET PDFS")
print("====================================")

for file in TARGET_FILENAMES:
    if file not in found_pdfs:
        print(f"❌ Skipping missing PDF: {file}")
        continue

    pdf_path = found_pdfs[file]

    print()
    print(f"📄 Parsing: {file}")
    print(f"   Path: {pdf_path}")

    rows = extract_data_from_pdf_auto(pdf_path)

    print(f"   Rows extracted: {len(rows)}")

    all_rows.extend(rows)


# ============================================================
# Write backfill CSV
# ============================================================

fieldnames = [
    "Source PDF",
    "Incident #",
    "Date",
    "Type",
    "Location",
    "Arrested",
    "Name",
    "DOB",
    "Charges"
]

with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(all_rows)


print()
print("====================================")
print("DONE")
print("====================================")
print(f"Backfill rows extracted: {len(all_rows)}")
print(f"CSV created at:")
print(OUTPUT_CSV)